In [1]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle
import matplotlib.pyplot as plt
import time
import os
import copy

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
df = pd.read_pickle("Data/df_data_2020_all_columns_08_11.pickle")

In [4]:
#with open("Data/df_all_columns_08_11.pickle", "wb") as f:
#    pickle.dump(df, f)

In [5]:
def contains_number(text):
    text = str(text)
    pattern = r'\d+(?:[,.]\d+)?'
    numbers = re.findall(pattern, text)
    if len(numbers) > 0:
        return True
    else:
        return False

In [6]:
#def function to fill array with sparseness values per column
def get_sparseness(df, columns, specification):
    sparseness_columns = []
    if specification == "multiple":
        for col in df.columns:
            if col.split(":")[0] == columns: 
                sparseness_columns.append(pd.isna(df[col]).sum() / len(df) * 100)
    elif specification == "single":
        for col in df.columns:
            if col in columns:
                sparseness_columns.append(pd.isna(df[col]).sum() / len(df) * 100)
    return sparseness_columns

In [7]:
def plot_sparseness(array, column_group_name):
    # count of the number of columns
    labels = [i for i in range(0, len(array))]
    if len(labels) > 1:
        # Create the bar chart
        plt.bar(labels, array)
        # Adding labels and a title
        plt.xlabel('columns for feature {}'.format(column_group_name))
        plt.ylabel('Sparseness %')
        plt.title('Sparseness per column for feature {}'.format(column_group_name))
        # Display the chart
        plt.show()

In [8]:
def create_fig_subplots(height, width, figsize, data_set, labels, title):
    fig, ax = plt.subplots(height, width, figsize=(figsize))
    fig.suptitle(title, fontsize=16)

    # Loop to plot the bar graphs
    for i in range(height):
        for j in range(width):
            data = data_set[i * width + j]
            ax[i, j].bar(x=[i for i in range(len(data))], height=data, color="grey")
            ax[i, j].title.set_text(labels[i*width + j])

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

In [9]:
def make_bar_graph(labels, data, label_rotation, figsize):
    # Create the bar chart
    plt.figure(figsize=figsize)
    plt.bar(labels, data, color="grey")
    # Rotate x-axis labels
    plt.xticks(rotation=label_rotation, ha="right")
    # Adding labels and a title
    plt.xlabel('features', fontsize = 17)
    plt.ylabel('Sparseness %', fontsize = 17)
    plt.title('Sparseness per feature', fontsize = 20)
    plt.show()

In [10]:
def get_unique_value_counts(df, columns):
    value_count_dict = {}
    for col in columns:
        labels = list(df[col].value_counts().keys().values)
        values = df[col].value_counts().values
        if values.sum() < len(df):
            remainder = len(df) - values.sum()
            labels.append("unknown")
            values = np.append(values, remainder)
        
        value_count_dict[col] = labels, list(values)
    return value_count_dict

GET STATISTICS ON THE CATEGORICAL COLUMNS

In [ ]:
#make a plot displaying the increasing sparsity per feature
#identify the variables for which there are multiple columns
column_groups = []
columns_with_multiple = []
for col in [col for col in df.columns if contains_number(col) == True]:
    column_groups.append(col.split(":")[0])
    columns_with_multiple.append(col)

column_groups = list(set(column_groups))

#get the sparseness per column group
sparseness_array_per_group = []
for column_group in column_groups:
    sparseness_array_per_group.append(get_sparseness(df, column_group, "multiple"))

create_fig_subplots(height=5, width=3, figsize=(12,10), data_set = sparseness_array_per_group, labels=column_groups, title = "Sparseness of features with multiple columns")

In [ ]:
#get the data for the remaining columns
remaining_cols = [col for col in df.columns if col not in columns_with_multiple]
data_remaning_cols = get_sparseness(df, columns=remaining_cols, specification="single")
make_bar_graph(data = data_remaning_cols, labels = remaining_cols, label_rotation=50, figsize=(12,10))

In [12]:
#let's get the value counts for all categorical columns
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
categorical_columns = ["country", "ca_type", "ca_activity", "cpv_code", "type_contract", "duration type", "ca_type_other", "ca_activity_other", "renewal", "procedure", "recurrent", "joint_procurement_involved", "central_purchasing", "eu_fund",
                       "awarded_contract", "framework or dynamic purchasing"]
plot_categorical = categorical_columns
plot_categorical.remove("cpv_code")

In [ ]:
#get the values
value_count_dict = get_unique_value_counts(df, plot_categorical)

fig, ax = plt.subplots(5, 3, figsize=(14, 22))
fig.suptitle("Distribution of values across categorical variables", fontsize=16)

# Loop to plot the bar graphs
keys_list = list(value_count_dict.keys())

for i in range(5):
    for j in range(3):
        key = keys_list[i*3 + j]
        data = value_count_dict[key][1]
        labels = value_count_dict[key][0]
        ax[i, j].bar(x=labels, height=data, color="grey")
        ax[i, j].title.set_text(keys_list[i*3 + j])
        # Rotate x-axis labels
        ax[i, j].set_xticklabels(labels, rotation=45, ha="right")
        # Adjust the horizontal spacing between subplots
plt.subplots_adjust(wspace=0.3, hspace=0.5)
# Adjust layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

In [14]:
def impute_columns_with_value(df, column, absent, replacement_value):
    count = 0
    values = []
    for i in tqdm(range(0, len(df))):
        value = df[column].iloc[i]
        if pd.isna(value) == absent:
            values.append(replacement_value)
            count += 1
        elif pd.isna(value) != absent:
            values.append(value)
        else:
            continue
    
    df[column] = values
    print("{} cases were imputed".format(count))

In [15]:
#Let's the columns for ca_other with ca_type with ca_type_other and ca_activity with ca_activity_other
def merge_categorical_columns(df, main_col, supplemental_col):
    count = 0
    values = []
    for i in tqdm(range(0, len(df))):
        main_value = df[main_col].iloc[i]
        supplemental_value = df[supplemental_col].iloc[i]
        if pd.isna(main_value) == True and pd.isna(supplemental_value) == False:
            values.append(supplemental_value)
        elif pd.isna(main_value) == False:
            values.append(main_value)
        else:
            values.append(np.nan)
    df[main_col] = values
    return print("{} cases were supplemented".format(count))

In [ ]:
impute_columns_with_value(df, "central_purchasing", True, "NO_CENTRAL_PURCHASING")
impute_columns_with_value(df, "joint_procurement_involved", True, "NO_JOINT_PROCUREMENT_INVOLVED")
impute_columns_with_value(df, "framework or dynamic purchasing", True, "no framework or dynamic purchasing")

merge_categorical_columns(df, "ca_type", "ca_type_other")
merge_categorical_columns(df, "ca_activity", "ca_activity_other")

imputed_merged_cat_cols = ["ca_type", "ca_activity", "central_purchasing", "joint_procurement_involved", "framework or dynamic purchasing"]
#get the values
value_count_dict = get_unique_value_counts(df,columns=imputed_merged_cat_cols)

In [ ]:
plt.figure(figsize=(12, 10))
ax1 = plt.subplot(2,3,1)
ax2 = plt.subplot(2,3,2)
ax3 = plt.subplot(2,3,3)
ax4 = plt.subplot(2,2,3)
ax5 = plt.subplot(2,2,4)
axes = [ax1, ax2, ax3, ax4, ax5]


for j in range(5):
    key = list(value_count_dict.keys())[j]
    data = value_count_dict[key][1]
    labels = value_count_dict[key][0]
    axes[j].bar(x=labels, height=data, color="grey")
    axes[j].title.set_text(keys_list[j])
    # Rotate x-axis labels
    if len(labels) < 4:
        axes[j].set_xticklabels(labels, rotation=40, fontsize = 10, ha="right")
    else:
        axes[j].set_xticklabels(labels, rotation=40, fontsize = 8, ha="right")
    # Adjust the horizontal spacing between subplots
plt.subplots_adjust(wspace=0.3, hspace=1)
# Adjust layout
plt.tight_layout(rect=[0, 1, 1, 0.95])

GET STATISTICS ON THE NUMERICAL COLUMNS

In [18]:
#get statistics on the numerical columns
#define numerical columns
numerical_columns = ["object_contract/val_estimated_total", "object_contract/val_total", "award_contract/val_estimated_total", "award_contract/val_total", "object_descr/duration",
                     "ac_price_weighting", "ac_weighting", "ac_cost/ac_weighting", "nb_tenders_received", "nb_tenders_received_sme", "lowest offer", "highest offer", "ac_price"]
#get the data and split based on column belonging to a group of columns or not
numerical_cols_single = [col for col in numerical_columns if col not in column_groups]
numerical_cols_multiple = [col for col in numerical_columns if col in column_groups]

In [19]:
def extract_numbers(text: str):
    text = str(text)
    pattern = r'\d+(?:[,.]\d+)?'
    numbers_list = re.findall(pattern, text)
    if len(numbers_list) > 0:
        for value in numbers_list:
            #the value has a comma, replace and multiply
            if "," in value:
                value = value.replace(",", ".")
                value = int(float(value) * 100)
            elif "." in value and int(float(value)) < 1:
                value = int(float(value)) * 100
            elif "." in value and int(float(value)) > 1:
                value = int(float(value))
            else:
                value = int(float(value))
    else:
        value = np.nan

    if value > 100:
        value = np.nan

    return value

In [20]:
def convert_to_days(df, value, i):
    average_days_per_month = 30.437
    if df["duration type"].iloc[i] == "MONTH":
        amount_in_days = int(int(value) * average_days_per_month)
    elif df["duration type"].iloc[i] == "DAY":
        amount_in_days = int(value)

    return amount_in_days

In [36]:
def get_numeric_data_multiple(df, features: list, specification = "not average", purpose = "produce statistics"):
    #create a list of columns on which we can aggregate
    columns_need_preprocessing = ['ac_price_weighting', 'ac_weighting', 'ac_cost/ac_weighting', "ac_price"]
    feature_columns_values = {}
    for feature in features:
        #get all columns matching the group identifier
        feature_columns = []
        for col in df.columns:
            if feature in col and "@" not in col:
                feature_columns.append(col)
            else:
                continue
        feature_columns_values[feature] = feature_columns
    
    aggregated_values = {}

    for feature in feature_columns_values.keys():
        values_per_feature = []
        for i in tqdm(range(0, len(df))):
            total = 0
            count_non_empty_cols = 0
            for col in feature_columns_values[feature]:
                if pd.isna(df[col].iloc[i]) == False:
                    value = df[col].iloc[i]
                    if feature in columns_need_preprocessing:
                        value = extract_numbers(value)
                    elif feature == "object_descr/duration":
                        value = convert_to_days(df, value, i)
                    else:
                        value = int(float(value))
                    
                    if pd.isna(value) == False:
                        total += value
                        count_non_empty_cols += 1
                    else:
                        continue
                    
                elif pd.isna(df[col].iloc[i]) == True:
                    continue
                else:
                    continue
            
            if total == 0 and purpose == "adjust df":
                total = np.nan
                values_per_feature.append(total)
            elif total == 0 and purpose == "produce statistics":
                continue
            elif total != 0 and pd.isna(total) == False and specification != "average":
                values_per_feature.append(total)
            elif total != 0 and pd.isna(total) == False and specification == "average":
                total = int(float(total / count_non_empty_cols))
                values_per_feature.append(total)
            
        aggregated_values[feature] = values_per_feature
        if purpose == "adjust df":
            df[feature] = values_per_feature
            df = df.drop(columns = feature_columns_values[feature])
            print("of feature '{}', {} columns were dropped".format(feature, len(feature_columns_values[feature])))
    if purpose == "produce statistics":
        return aggregated_values
    else:
        return "Dataframe was adjusted"

In [37]:
def make_boxplots(dict_feature_values):
    plt.figure(figsize=(12, 10))
    ax1 = plt.subplot(2,3,1)
    ax2 = plt.subplot(2,3,2)
    ax3 = plt.subplot(2,3,3)
    ax4 = plt.subplot(2,4,3)
    ax5 = plt.subplot(2,4,4)
    ax6 = plt.subplot(2,4,5)
    ax7 = plt.subplot(2,4,6)
    axes = [ax1, ax2, ax3, ax4, ax5]
    fig.suptitle("Boxplot for numeric features", fontsize=16)
    
    for j in range(5):
        feature = list(dict_feature_values.keys())[j]
        data = dict_feature_values[feature]
        axes[j].boxplot(data)
        axes[j].title.set_text(feature)

    plt.tight_layout()
    plt.show()

In [38]:
def get_statistics_df(dict_feature_values):
    
    df_numeric_describe = pd.DataFrame()
    for feature in dict_feature_values:
        df_numeric = pd.DataFrame(dict_feature_values[feature])
        df_numeric_describe[feature] = df_numeric.describe()

    return df_numeric_describe

In [ ]:
#split columns depending on averaging or adding values
numerical_cols_multiple_money = numerical_cols_multiple[:3]
numerical_cols_multiple_weighted = numerical_cols_multiple[3:]

#!!UNCOMMENT LINES BELOW TO RUN THE FUNCTIONS AGAIN!!
#numerical_cols_multiple_money_arrays = get_numeric_data_multiple(df, numerical_cols_multiple_money)
#numerical_cols_multiple_weighted_arrays = get_numeric_data_multiple(df, numerical_cols_multiple_weighted, "average")

#run the code for the purpose of modifying the dataframe instead of statistics
numerical_cols_multiple_money_arrays = get_numeric_data_multiple(df, numerical_cols_multiple_money, purpose = "adjust df")
numerical_cols_multiple_weighted_arrays = get_numeric_data_multiple(df, numerical_cols_multiple_weighted, "average", purpose = "adjust df")

In [ ]:
numerical_cols_multiple_money[0]

In [260]:
with open("Data/numerical_cols_multiple_money_arrays.pickle", "rb") as f:
    numerical_cols_multiple_money_arrays = pickle.load(f)

with open("Data/numerical_cols_multiple_weighted_arrays.pickle", "rb") as f:
    numerical_cols_multiple_weighted_arrays = pickle.load(f)

In [261]:
#combine all numeric features
aggregated_values_numeric = numerical_cols_multiple_money_arrays | numerical_cols_multiple_weighted_arrays

In [ ]:
df_statistics_numeric = get_statistics_df(aggregated_values_numeric)
df_statistics_numeric

In [264]:
def make_boxplots(dict_feature_values):
    plt.figure(figsize=(12, 10))

    # Create the top row of subplots
    ax1 = plt.subplot(2, 4, 1)
    ax2 = plt.subplot(2, 4, 2)
    ax3 = plt.subplot(2, 4, 3)

    # Create the bottom row of subplots
    ax4 = plt.subplot(2, 4, 5)
    ax5 = plt.subplot(2, 4, 6)
    ax6 = plt.subplot(2, 4, 7)
    ax7 = plt.subplot(2, 4, 8)

    axes = [ax1, ax2, ax3, ax4, ax5, ax6, ax7]
    fig.suptitle("Boxplot for numeric features", fontsize=16)

    for j in range(7):
        feature = list(dict_feature_values.keys())[j]
        data = dict_feature_values[feature]
        axes[j].boxplot(data)
        axes[j].title.set_text(feature)

    plt.tight_layout()
    plt.show()

In [ ]:
make_boxplots(aggregated_values_numeric)

In [43]:
def get_columns(df, features):
    feature_columns_values = {}
    for feature in features:
        feature_columns = []
        for col in df.columns:
            if feature in col.split(":")[0]:
                feature_columns.append(col)
            else:
                continue
        feature_columns_values[feature] = feature_columns
    return feature_columns_values

In [40]:
#aggregate all text columns of a given feature
def get_text_muliple_feature(df, features, purpose = "produce statistics"):
    feature_columns_values_text = get_columns(df, features)
    aggregated_values_text = {}

    for feature in feature_columns_values_text.keys():
        values_per_feature = []
        for i in tqdm(range(0, len(df))):
            text_row = ""
            for col in feature_columns_values_text[feature]:
                if pd.isna(df[col].iloc[i]) == False:
                    text_row += (" " +str(df[col].iloc[i]))
                elif pd.isna(df[col].iloc[i]) == True:
                    continue
                
            if len(text_row) == 0 and purpose == "produce statistics":
                continue
            elif len(text_row) == 0 and purpose == "adjust df":
                values_per_feature.append(np.nan)
            elif len(text_row) > 0:
                values_per_feature.append(text_row)
            else:
                print("strange case this is")
        
        aggregated_values_text[feature] = values_per_feature
    
    return aggregated_values_text

In [41]:
def get_text_single_feature(df, columns):
    dict_of_text_values = {}
    for col in columns:
        text_per_col = []
        for i in range(0, len(df)):
            cell_text = str(df[col].iloc[i])
            text_per_col.append(cell_text)
        dict_of_text_values[col] = text_per_col
    return dict_of_text_values

In [42]:
#produce describtive statistics for the aggregated text columns
def convert_text_to_array(aggregated_values_text):
    word_counts_per_feature_dict = {}
    for feature in aggregated_values_text:
        feature_word_count = []
        for text in aggregated_values_text[feature]:
            word_count = len(text.split())
            feature_word_count.append(word_count)
        word_counts_per_feature_dict[feature] = feature_word_count
    return word_counts_per_feature_dict

In [44]:
#define the text columns
text_features = ["object_descr/title", "ac_cost/ac_criterion", "ac_criterion"]
text_columns_single = ["short_descr", "object_contract/title"]
text_columns_with_multiple = [col for col in columns_with_multiple if col.split(":")[0] in text_features]
text_columns = text_columns_with_multiple + text_columns_single

In [ ]:
#!!UNCOMMENT TO RUN THE CODE AGAIN!!
aggregated_values_text_single = get_text_single_feature(df, text_columns_single, purpose = "adjust df")
aggregated_values_text_multiple = get_text_muliple_feature(df, text_features, purpose = "adjust df")

In [258]:
#with open("Data/aggregated_values_text_single.pickle", "wb") as f:
#    pickle.dump(aggregated_values_text_single, f)

#with open("Data/aggregated_values_text_multiple.pickle", "wb") as f:
#    pickle.dump(aggregated_values_text_multiple, f)

In [ ]:
with open("Data/aggregated_values_text_single.pickle", "rb") as f:
    aggregated_values_text_single = pickle.load(f)

with open("Data/aggregated_values_text_multiple.pickle", "rb") as f:
    aggregated_values_text_multiple = pickle.load(f)

In [ ]:
#combine all text features
aggregated_values_text = aggregated_values_text_multiple | aggregated_values_text_single
word_counts_per_feature_dict = convert_text_to_array(aggregated_values_text)
df_statistics_text = get_statistics_df(word_counts_per_feature_dict)
df_statistics_text

---------------------------------------------------------------------------------------------------------------------------------------
DROPPING UNSUITABLE ROWS
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

Preprocessing steps taken so far: <br>
1) Merging columns of features having multiple columns <br>
2) Supplementing features with other features (feature engineering) <br>
3) 